# S-Fig 10 — Reliability Diagrams (maybe)

Calibration check: predicted probability vs actual fraction positive.  
**Source**: collected parquets at context=240m  
**Tasks**: main tasks

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root is correct ─────────────────────────────────────
_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD     = "lstm"
SPLIT    = "test"
# 3 representative contexts overlaid per panel (30s shortest, 120m mid, 240m longest)
CONTEXTS = ["30s", "120m", "240m"]
K        = 5   # windows aggregated per subject (matches paper deployment)
TASKS    = [t for t in MAIN_TASKS if t != "age_class"]   # age is 3-class, skip
N_COLS   = 2
N_ROWS   = (len(TASKS) + N_COLS - 1) // N_COLS
ROW_H    = 2.6

pqs = {t: load_parquets("phase0_v3", t, HEAD, SPLIT) for t in TASKS}
print("Loaded tasks:", list(pqs.keys()))

In [ ]:
labels = [chr(97 + i) for i in range(len(TASKS))]
n_last = len(TASKS) % N_COLS or N_COLS
n_full = len(TASKS) // N_COLS

mosaic = []
for row in range(n_full):
    rl = labels[row * N_COLS : (row + 1) * N_COLS]
    mosaic.append([l for l in rl for _ in range(2)])
if n_last < N_COLS:
    pad = N_COLS - n_last; ll = labels[n_full * N_COLS:]
    mosaic.append(["."] * pad + [l for l in ll for _ in range(2)] + ["."] * pad)

fig, axd = plt.subplot_mosaic(mosaic, figsize=(FULL_W, N_ROWS * ROW_H))

for i, (lbl, task) in enumerate(zip(labels, TASKS)):
    panels.reliability_panel(axd[lbl], pqs[task], contexts=CONTEXTS, k=K)
    axd[lbl].set_title(TASK_LABEL.get(task, task), fontsize=8)
    add_panel_label(axd[lbl], f"({lbl})")

fig.tight_layout(h_pad=1.2, w_pad=1.0)
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
save_figure(fig, FINAL_OUT, "sfig10_reliability")
print("Saved →", FINAL_OUT / "sfig10_reliability.pdf")